# CatBoost для детекции мошенничества по транзакциям

Обучение классической модели CatBoost на датасете `card_transaction.v1.csv` (IBM Credit Card Transactions).

**Цель:** бинарная классификация `Is Fraud?` (Yes/No).

Пайплайн:
1. Загрузка CSV (с оптимизацией dtypes, т.к. файл ~2.3 ГБ).
2. Очистка и инженерия признаков.
3. Train/val/test сплит по времени.
4. Обучение `CatBoostClassifier` с балансировкой классов.
5. Оценка качества (ROC-AUC, PR-AUC, classification report, confusion matrix).
6. Анализ важности признаков.

In [7]:
# import sys
# !{sys.executable} -m pip install --user catboost

# # После установки, добавим путь в sys.path
# import site
# user_site = site.getusersitepackages()
# if user_site not in sys.path:
#     sys.path.insert(0, user_site)
#     print(f"Добавлен путь: {user_site}")

# # Теперь должно работать
# import catboost
# print(f"CatBoost version: {catboost.__version__}")

In [31]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

from catboost import CatBoostClassifier, Pool

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")

RANDOM_STATE = 42
DATA_PATH = "card_transaction.v1.csv"

## 1. Загрузка данных

Файл большой (~2.3 ГБ). Указываем dtypes сразу и читаем чанками — это сильно экономит память.
При необходимости можно ограничить количество строк параметром `NROWS`.

In [44]:
NROWS = None  # поставить число (например 5_000_000), если хочется быстрее

dtypes = {
    "User": "int32",
    "Card": "int16",
    "Year": "int16",
    "Month": "int8",
    "Day": "int8",
    "Time": "string",
    "Amount": "string",
    "Use Chip": "category",
    "Merchant Name": "string",
    "Merchant City": "category",
    "Merchant State": "category",
    "Zip": "string",
    "MCC": "int32",
    "Errors?": "string",
    "Is Fraud?": "category",
}

df = pd.read_csv(DATA_PATH, dtype=dtypes, nrows=NROWS)
print(f"Shape: {df.shape}")
df.head()

Shape: (24386900, 15)


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,<NA>,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,<NA>,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,<NA>,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,<NA>,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,<NA>,No


In [45]:
print(df.info(memory_usage="deep"))
print("\nДоля мошеннических транзакций:")
print(df["Is Fraud?"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 15 columns):
 #   Column          Dtype   
---  ------          -----   
 0   User            int32   
 1   Card            int16   
 2   Year            int16   
 3   Month           int8    
 4   Day             int8    
 5   Time            string  
 6   Amount          string  
 7   Use Chip        category
 8   Merchant Name   string  
 9   Merchant City   category
 10  Merchant State  category
 11  Zip             string  
 12  MCC             int32   
 13  Errors?         string  
 14  Is Fraud?       category
dtypes: category(4), int16(2), int32(2), int8(2), string(5)
memory usage: 7.9 GB
None

Доля мошеннических транзакций:
Is Fraud?
No     0.99878
Yes    0.00122
Name: proportion, dtype: float64


## 2. Препроцессинг и фичи

- `Amount`: убираем `$` и приводим к float.
- `Time`: парсим часы/минуты, добавляем фичи времени суток.
- `Errors?`: бинарная фича «была ошибка».
- Делаем единый timestamp для временного сплита.
- Категориалки оставляем как `category` — CatBoost будет работать с ними нативно.

In [46]:
df["Amount"] = df["Amount"].str.replace("$", "", regex=False).astype("float32")

time_split = df["Time"].str.split(":", expand=True)
df["Hour"] = time_split[0].astype("int8")
df["Minute"] = time_split[1].astype("int8")

df["HasError"] = df["Errors?"].notna().astype("int8")

df["Timestamp"] = pd.to_datetime(
    dict(year=df["Year"], month=df["Month"], day=df["Day"],
         hour=df["Hour"], minute=df["Minute"])
)

df["DayOfWeek"] = df["Timestamp"].dt.dayofweek.astype("int8")
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype("int8")
df["IsNight"] = ((df["Hour"] < 6) | (df["Hour"] >= 22)).astype("int8")

df["Zip"] = df["Zip"].fillna("UNK").astype("category")
df["Merchant Name"] = df["Merchant Name"].astype("category")

df["target"] = (df["Is Fraud?"] == "Yes").astype("int8")

df = df.drop(columns=["Time", "Errors?", "Is Fraud?"])
gc.collect()
df.head()

,User,Card,Year,Month,Day,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Hour,Minute,HasError,Timestamp,DayOfWeek,IsWeekend,IsNight,target
0,0,0,2002,9,1,134.089996,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,6,21,0,2002-09-01 06:21:00,6,1,0,0
1,0,0,2002,9,1,38.480000,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,6,42,0,2002-09-01 06:42:00,6,1,0,0
2,0,0,2002,9,2,120.339996,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,6,22,0,2002-09-02 06:22:00,0,0,0,0
3,0,0,2002,9,2,128.949997,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,17,45,0,2002-09-02 17:45:00,0,0,0,0
4,0,0,2002,9,3,104.709999,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,6,23,0,2002-09-03 06:23:00,1,0,0,0


In [47]:
df = df.sort_values(["User", "Timestamp"], kind="stable").reset_index(drop=True)
g_user = df.groupby("User", sort=False)

df["TimeSincePrevUser"] = (
    g_user["Timestamp"].diff().dt.total_seconds().astype("float32")
)
df["TimeSincePrevUser"] = df["TimeSincePrevUser"].clip(upper=86400 * 30)

df["UserTxnRank"] = g_user.cumcount().astype("int32")

cum_sum = g_user["Amount"].cumsum()
cum_sum_prev = (cum_sum - df["Amount"]).astype("float32")
denom = df["UserTxnRank"].replace(0, np.nan)
df["UserCumMean"] = (cum_sum_prev / denom).astype("float32")

cum_sum_sq = g_user["Amount"].transform(lambda s: (s ** 2).cumsum())
cum_sum_sq_prev = (cum_sum_sq - df["Amount"] ** 2).astype("float64")
mean_sq = cum_sum_sq_prev / denom
var = (mean_sq - df["UserCumMean"].astype("float64") ** 2).clip(lower=0)
df["UserCumStd"] = np.sqrt(var).astype("float32")

df["AmountVsUserMean"] = (df["Amount"] - df["UserCumMean"]).astype("float32")
df["AmountZScoreUser"] = (
    df["AmountVsUserMean"] / (df["UserCumStd"] + 1e-3)
).astype("float32")

df["SameMerchantAsPrev"] = (
    g_user["Merchant Name"].shift(1) == df["Merchant Name"]
).astype("int8")
df["SameCityAsPrev"] = (
    g_user["Merchant City"].shift(1) == df["Merchant City"]
).astype("int8")
df["SameStateAsPrev"] = (
    g_user["Merchant State"].shift(1) == df["Merchant State"]
).astype("int8")

seen_count = df.groupby(["User", "Merchant Name"], sort=False).cumcount()
df["IsNewMerchantForUser"] = (seen_count == 0).astype("int8")

df = df.sort_values(["User", "Card", "Timestamp"], kind="stable").reset_index(drop=True)
df["TimeSincePrevCard"] = (
    df.groupby(["User", "Card"], sort=False)["Timestamp"]
    .diff().dt.total_seconds().astype("float32")
)
df["TimeSincePrevCard"] = df["TimeSincePrevCard"].clip(upper=86400 * 30)

df = df.sort_values("Timestamp", kind="stable").reset_index(drop=True)

new_feats = [
    "TimeSincePrevUser", "TimeSincePrevCard", "UserTxnRank",
    "UserCumMean", "UserCumStd", "AmountVsUserMean", "AmountZScoreUser",
    "SameMerchantAsPrev", "SameCityAsPrev", "SameStateAsPrev",
    "IsNewMerchantForUser",
]
print("Добавлены фичи:", new_feats)
print(df[new_feats].describe())

/tmp/job-3979147/ipykernel_107798/2004602429.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  seen_count = df.groupby(["User", "Merchant Name"], sort=False).cumcount()


Добавлены фичи: ['TimeSincePrevUser', 'TimeSincePrevCard', 'UserTxnRank', 'UserCumMean', 'UserCumStd', 'AmountVsUserMean', 'AmountZScoreUser', 'SameMerchantAsPrev', 'SameCityAsPrev', 'SameStateAsPrev', 'IsNewMerchantForUser']
       TimeSincePrevUser  TimeSincePrevCard   UserTxnRank   UserCumMean  \
count       2.438490e+07       2.438076e+07  2.438690e+07  2.438490e+07   
mean        2.883695e+04       6.359098e+04  1.080618e+04  4.460839e+01   
std         3.812836e+04       9.922977e+04  9.819142e+03  1.810071e+01   
min         0.000000e+00       0.000000e+00  0.000000e+00 -1.000000e+02   
25%         2.760000e+03       5.640000e+03  3.866000e+03  3.183482e+01   
50%         1.392000e+04       3.096000e+04  8.273000e+03  4.169353e+01   
75%         4.374000e+04       8.238000e+04  1.474400e+04  5.420879e+01   
max         2.592000e+06       2.592000e+06  8.235400e+04  5.064750e+03   

         UserCumStd  AmountVsUserMean  AmountZScoreUser  SameMerchantAsPrev  \
count  2.438490e+07

## 3. Train/Val/Test сплит по времени

В fraud-задачах временной сплит честнее случайного: модель не подсматривает в будущее.

In [13]:
# df = df.sort_values("Timestamp").reset_index(drop=True)

# n = len(df)
# train_end = int(n * 0.7)
# val_end = int(n * 0.85)

# train_df = df.iloc[:train_end]
# val_df = df.iloc[train_end:val_end]
# test_df = df.iloc[val_end:]

# print(f"Train:  {len(train_df):>10,}  fraud rate: {train_df['target'].mean():.4%}")
# print(f"Val:    {len(val_df):>10,}  fraud rate: {val_df['target'].mean():.4%}")
# print(f"Test:   {len(test_df):>10,}  fraud rate: {test_df['target'].mean():.4%}")

# print(f"\nTrain period: {train_df['Timestamp'].min()} — {train_df['Timestamp'].max()}")
# print(f"Val period:   {val_df['Timestamp'].min()} — {val_df['Timestamp'].max()}")
# print(f"Test period:  {test_df['Timestamp'].min()} — {test_df['Timestamp'].max()}")

In [48]:
import pandas as pd

def split_by_time_per_user(
    df,
    user_col="user",
    time_col="timestamp",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

    df = df.sort_values([user_col, time_col]).reset_index(drop=True)

    train_parts = []
    val_parts = []
    test_parts = []

    for _, group in df.groupby(user_col, sort=False):
        n = len(group)

        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)

        # чтобы не развалиться на маленьких группах
        if train_end == 0:
            train_end = min(1, n)
        if val_end <= train_end and n - train_end > 1:
            val_end = train_end + 1

        train_parts.append(group.iloc[:train_end])
        val_parts.append(group.iloc[train_end:val_end])
        test_parts.append(group.iloc[val_end:])

    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)

    return train_df, val_df, test_df

In [49]:
df = df.sort_values("Timestamp").reset_index(drop=True)
df.head(2)

,User,Card,Year,Month,Day,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Hour,Minute,HasError,Timestamp,DayOfWeek,IsWeekend,IsNight,target,TimeSincePrevUser,UserTxnRank,UserCumMean,UserCumStd,AmountVsUserMean,AmountZScoreUser,SameMerchantAsPrev,SameCityAsPrev,SameStateAsPrev,IsNewMerchantForUser,TimeSincePrevCard
0,791,1,1991,1,2,68.0,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,7,10,0,1991-01-02 07:10:00,2,0,0,0,NaN,0,NaN,NaN,NaN,NaN,0,0,0,1,NaN
1,791,1,1991,1,2,-68.0,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,7,17,0,1991-01-02 07:17:00,2,0,0,0,420.0,1,68.0,0.0,-136.0,-136000.0,1,1,1,0,420.0


In [50]:
train_df, val_df, test_df = split_by_time_per_user(
    df,
    user_col="User",
    time_col="Timestamp",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15
)

print(len(train_df), len(val_df), len(test_df))
print(len(train_df) / len(df), len(val_df) / len(df), len(test_df) / len(df))



17069880 3657092 3659928
0.6999610446592228 0.14996133169857587 0.15007762364220134


In [17]:
# feature_cols = [
#     "User", "Card", "Month", "Day", "Hour", "Minute",
#     "DayOfWeek", "IsWeekend", "IsNight",
#     "Amount", "Use Chip",
#     "Merchant Name", "Merchant City", "Merchant State", "Zip",
#     "MCC", "HasError",
# ]
# cat_features = [
#     "User", "Card", "Use Chip",
#     "Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC",
# ]

In [51]:
   # "User",
feature_cols = [
     "Card", "Month", "Day", "Minute",
    "DayOfWeek", "IsWeekend", "IsNight",
    "Amount", "Use Chip",
    "Merchant Name", "Merchant City", "Merchant State", "Zip",
    "MCC", "HasError",
    "TimeSincePrevUser", "TimeSincePrevCard", "UserTxnRank",
    "UserCumMean", "UserCumStd",
    "AmountVsUserMean", "AmountZScoreUser",
    "SameMerchantAsPrev", "SameCityAsPrev", "SameStateAsPrev",
    "IsNewMerchantForUser",
]
cat_features = [
     "Card", "Use Chip",
    "Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC",
]

In [52]:
for col in cat_features:
    train_df[col] = train_df[col].astype(str)
    val_df[col]   = val_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)

X_train, y_train = train_df[feature_cols], train_df["target"]
X_val,   y_val   = val_df[feature_cols],   val_df["target"]
X_test,  y_test  = test_df[feature_cols],  test_df["target"]

del df, train_df, val_df, test_df
gc.collect()

0

## 4. Обучение CatBoost

Сильный дисбаланс классов → используем `auto_class_weights="Balanced"`.

In [ ]:
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool   = Pool(X_val,   y_val,   cat_features=cat_features)
test_pool  = Pool(X_test,  y_test,  cat_features=cat_features)

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="F1",
    custom_metric=["AUC", "PRAUC", "F1", "Precision", "Recall"],
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    od_type="Iter",
    od_wait=150,
    task_type="CPU",
    thread_count=-1,
    verbose=100,
)

model.fit(train_pool, eval_set=val_pool, use_best_model=True)
print(f"\nЛучшая итерация: {model.get_best_iteration()}")

0:	learn: 0.9312587	test: 0.9565280	best: 0.9565280 (0)	total: 9.99s	remaining: 5h 32m 49s
100:	learn: 0.9591267	test: 0.9738765	best: 0.9739662 (99)	total: 18m 56s	remaining: 5h 56m 10s


## 5. Оценка качества

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(model, pool, y_true, name="set"):
    proba = model.predict_proba(pool)[:, 1]
    pred = (proba >= 0.5).astype(int)
    
    # Основные метрики
    auc = roc_auc_score(y_true, proba)
    pr_auc = average_precision_score(y_true, proba)
    precision = precision_score(y_true, pred)
    recall = recall_score(y_true, pred)
    f1 = f1_score(y_true, pred)
    
    print(f"\n=== {name} ===")
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    print(classification_report(y_true, pred, digits=4))
    return proba, pred

val_proba,  val_pred  = evaluate(model, val_pool,  y_val,  "Validation")
test_proba, test_pred = evaluate(model, test_pool, y_test, "Test")

In [ ]:
train_proba,  train_pred  = evaluate(model, train_pool,  y_train,  "Validation")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
axes[0].set_title("Confusion Matrix (Test, threshold=0.5)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_proba):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("ROC Curve (Test)")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, test_proba)
axes[2].plot(rec, prec, label=f"AP = {average_precision_score(y_test, test_proba):.4f}")
axes[2].set_title("Precision-Recall Curve (Test)")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Важность признаков

In [ ]:
importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False)

print(importances.to_string(index=False))

plt.figure(figsize=(9, 6))
sns.barplot(data=importances, x="importance", y="feature", palette="viridis")
plt.title("CatBoost Feature Importance")
plt.tight_layout()
plt.show()

## 7. Подбор порога и сохранение модели

Из-за сильного дисбаланса порог 0.5 редко оптимален — подберём по F1 на валидации.

In [ ]:
prec_v, rec_v, thr_v = precision_recall_curve(y_val, val_proba)
f1_v = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-12)
best_idx = int(np.nanargmax(f1_v[:-1]))
best_thr = float(thr_v[best_idx])
print(f"Лучший порог по F1 на val: {best_thr:.4f} (F1={f1_v[best_idx]:.4f})")

test_pred_opt = (test_proba >= best_thr).astype(int)
print("\nTest @ optimal threshold:")
print(classification_report(y_test, test_pred_opt, digits=4))

model.save_model("catboost_fraud_without_user.cbm")
print("\nМодель сохранена в catboost_fraud.cbm")

In [ ]:

def evaluate(model, pool, y_true, name="set"):
    proba = model.predict_proba(pool)[:, 1]
    pred = (proba >= 0.9831).astype(int)
    
    # Основные метрики
    auc = roc_auc_score(y_true, proba)
    pr_auc = average_precision_score(y_true, proba)
    precision = precision_score(y_true, pred)
    recall = recall_score(y_true, pred)
    f1 = f1_score(y_true, pred)
    
    print(f"\n=== {name} ===")
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    print(classification_report(y_true, pred, digits=4))
    return proba, pred

val_proba,  val_pred  = evaluate(model, val_pool,  y_val,  "Validation")

In [ ]:
auc = roc_auc_score(y_test, test_proba)
pr_auc = average_precision_score(y_test, test_proba)
precision = precision_score(y_test, test_pred_opt)
recall = recall_score(y_test, test_pred_opt)
f1 = f1_score(y_test, test_pred_opt)
    

print(f"ROC-AUC:   {auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")